In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q ultralytics opencv-python-headless pandas pyyaml


In [ ]:
from pathlib import Path
import os

candidatos = [
    Path('/content/ENTREGA_GITHUB_ONIBUS_CHUVA_SP'),
    Path('/content/pacote_onibus_chuva_colab'),
    Path('/content/drive/MyDrive/ENTREGA_GITHUB_ONIBUS_CHUVA_SP'),
    Path('/content/drive/MyDrive/pacote_onibus_chuva_colab'),
    Path('/content/drive/MyDrive/projeto_onibus_chuva'),
]
projeto = next((p for p in candidatos if (p / 'models' / 'best.pt').exists()), None)
if projeto is None:
    encontrados = list(Path('/content').glob('**/best.pt'))
    projeto = next((p.parent.parent for p in encontrados if (p.parent.name == 'models')), None)
if projeto is None:
    raise FileNotFoundError('best.pt não encontrado')
modelo_path = projeto / 'models' / 'best.pt'
data_path = projeto / 'dataset' / 'data.yaml'
video_path = projeto / 'amostras' / 'video_ponto_onibus.mp4'
saidas = projeto / 'saidas'
saidas.mkdir(parents=True, exist_ok=True)
video_saida = saidas / 'avaliacao_demo.mp4'
csv_saida = saidas / 'avaliacao_contagens.csv'
print(projeto)
print(modelo_path)
print(data_path)
print(video_path)


In [ ]:
from ultralytics import YOLO

modelo = YOLO(str(modelo_path))
print(modelo.names)


In [ ]:
metricas = modelo.val(
    data=str(data_path),
    split='test',
    imgsz=640,
    plots=True,
    project=str(saidas),
    name='avaliacao_metricas',
    exist_ok=True,
)
print(f'Precisão: {metricas.box.mp * 100:.2f}%')
print(f'Recall: {metricas.box.mr * 100:.2f}%')
print(f'mAP50: {metricas.box.map50 * 100:.2f}%')
print(f'mAP50-95: {metricas.box.map * 100:.2f}%')


In [ ]:
import subprocess
import sys

comando = [
    sys.executable,
    str(projeto / 'src' / 'detectar_tempo_real.py'),
    '--pesos', str(modelo_path),
    '--fonte', str(video_path),
    '--anonimizar',
    '--sem-janela',
    '--salvar-video', str(video_saida),
    '--csv', str(csv_saida),
    '--conf', '0.35',
    '--imgsz', '640',
    '--intervalo', '1',
]
execucao = subprocess.run(comando, cwd=str(projeto), text=True, capture_output=True)
print(execucao.stdout)
if execucao.returncode != 0:
    print(execucao.stderr)
    raise RuntimeError('Falha na execução')


In [ ]:
from IPython.display import Video, display

display(Video(str(video_saida), embed=True, width=900))


In [ ]:
import pandas as pd

dados = pd.read_csv(csv_saida)
display(dados.head(10))
print(f'Linhas: {len(dados)}')
print(csv_saida)
